# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfurqan1/FlyRank-MachineLearning/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Chosen methods: Logistic Regression and Random Forest

The outcome is binary: 1 means April impressions fell by more than 15% from March, and 0 means they did not. The models produce scores used to rank content items for review, so this is classification used for ranking.

Logistic Regression is the primary model because it is reproducible and interpretable. Random Forest is included as a nonlinear challenger but will be preferred only if it provides a meaningful and stable improvement.

Precision@50 is the primary metric established in Week 2, while Precision@20 remains a secondary operational metric from Week 4. Both models and the frozen baseline are evaluated on the same client-grouped folds.

The scores support prioritization but are not treated as calibrated probabilities. The analysis shows association rather than causation, and June 2026 remains sealed.

---



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

March 2026 is the observation window, and April 2026 supplies the label. One row represents one pseudonymized client-content pair. The label is 1 when April impressions are more than 15% below March impressions.

I use five-fold `GroupKFold` validation grouped by client. Every client belongs to one fold, so the model is never validated on content from a client it saw during training. `GroupKFold` was chosen because the preliminary stratified grouped split produced severely unequal fold sizes.

The frozen baseline and Logistic Regression will be evaluated on the same rows in each fold using Precision@20 and Precision@50. Fold base rates will also be reported because client outcome patterns may differ. June 2026 remains sealed.

In [8]:
import getpass

import duckdb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import GroupKFold

RANDOM_SEED = 42
N_SPLITS = 5
DECLINE_THRESHOLD = -0.15
OUTCOME_MULTIPLIER = 1 + DECLINE_THRESHOLD

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

if not hf_token or not hf_token.startswith("hf_"):
    raise ValueError("A valid Hugging Face READ token is required.")

con = duckdb.connect()
safe_token = hf_token.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET flyrank_hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

del hf_token, safe_token

WAREHOUSE_ROOT = "hf://datasets/FlyRank/internship-warehouse"
MARCH_FACT = f"{WAREHOUSE_ROOT}/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_FACT = f"{WAREHOUSE_ROOT}/fact_content_daily_performance/month=2026-04/*.parquet"

analysis_df = con.execute(
    f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0) AS march_avg_position
        FROM read_parquet('{MARCH_FACT}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM read_parquet('{APRIL_FACT}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_impressions,
        m.march_clicks,
        m.march_avg_position,
        100.0 * m.march_clicks
            / NULLIF(m.march_impressions, 0) AS march_ctr,
        a.april_impressions,
        CASE
            WHEN a.april_impressions
                < {OUTCOME_MULTIPLIER} * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march AS m
    INNER JOIN april AS a
        USING (client_hash_id, content_hash_id)
    WHERE m.march_impressions > 0
    """
).df()

assert not analysis_df.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()
assert analysis_df["march_impressions"].gt(0).all()
assert analysis_df["is_declining_label"].isin([0, 1]).all()

splitter = GroupKFold(n_splits=N_SPLITS)
analysis_df["validation_fold"] = -1

for fold_number, (_, validation_index) in enumerate(
    splitter.split(
        X=np.zeros(len(analysis_df)),
        y=analysis_df["is_declining_label"],
        groups=analysis_df["client_hash_id"],
    ),
    start=1,
):
    row_index = analysis_df.iloc[validation_index].index
    analysis_df.loc[row_index, "validation_fold"] = fold_number

assert analysis_df["validation_fold"].between(1, N_SPLITS).all()

client_fold_counts = (
    analysis_df.groupby("client_hash_id")["validation_fold"].nunique()
)
assert client_fold_counts.eq(1).all()

for fold_number in range(1, N_SPLITS + 1):
    training_clients = set(
        analysis_df.loc[
            analysis_df["validation_fold"] != fold_number,
            "client_hash_id",
        ]
    )
    validation_clients = set(
        analysis_df.loc[
            analysis_df["validation_fold"] == fold_number,
            "client_hash_id",
        ]
    )
    validation_labels = analysis_df.loc[
        analysis_df["validation_fold"] == fold_number,
        "is_declining_label",
    ]

    assert training_clients.isdisjoint(validation_clients)
    assert validation_labels.nunique() == 2

fold_summary = (
    analysis_df.groupby("validation_fold")
    .agg(
        validation_rows=("content_hash_id", "size"),
        validation_clients=("client_hash_id", "nunique"),
        decline_base_rate=("is_declining_label", "mean"),
    )
    .reset_index()
)

fold_summary["decline_base_rate"] = (
    fold_summary["decline_base_rate"] * 100
).round(1)

fold_size_ratio = (
    fold_summary["validation_rows"].max()
    / fold_summary["validation_rows"].min()
)

print(f"Modeling rows: {len(analysis_df):,}")
print(f"Clients: {analysis_df['client_hash_id'].nunique():,}")
print(
    "Overall decline base rate: "
    f"{analysis_df['is_declining_label'].mean():.1%}"
)
print(f"Split method: GroupKFold ({N_SPLITS} folds)")
print(f"Largest-to-smallest fold ratio: {fold_size_ratio:.2f}")
print("Client overlap check: passed")
print("June 2026 loaded: no")
print(f"scikit-learn version: {sklearn.__version__}")

display(fold_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 158,549
Clients: 46
Overall decline base rate: 51.1%
Split method: GroupKFold (5 folds)
Largest-to-smallest fold ratio: 1.00
Client overlap check: passed
June 2026 loaded: no
scikit-learn version: 1.6.1


,validation_fold,validation_rows,validation_clients,decline_base_rate
0,1,31712,8,46.3
1,2,31708,8,67.8
2,3,31707,10,35.6
3,4,31706,11,54.4
4,5,31716,9,51.3


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


The models use six March-only features: log impressions, log clicks, average position, CTR, a zero-click indicator, and the Week 4 eligibility indicator. Counts are log-transformed to reduce the influence of extreme values.

Logistic Regression is the primary model. A restricted Random Forest is included as a nonlinear challenger. Both models and the frozen baseline are evaluated on the same validation rows using Precision@20 and Precision@50. Complexity is useful only if the forest improves performance consistently.

In [9]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

analysis_df["log_march_impressions"] = np.log1p(
    analysis_df["march_impressions"]
)
analysis_df["log_march_clicks"] = np.log1p(
    analysis_df["march_clicks"]
)
analysis_df["zero_clicks"] = (
    analysis_df["march_clicks"] == 0
).astype(int)
analysis_df["baseline_eligible"] = (
    (analysis_df["march_impressions"] >= 100)
    & (analysis_df["march_clicks"] == 0)
    & (analysis_df["march_avg_position"] > 0)
    & (analysis_df["march_avg_position"] <= 20)
).astype(int)
analysis_df["baseline_score"] = np.where(
    analysis_df["baseline_eligible"] == 1,
    analysis_df["march_impressions"],
    0,
)

FEATURE_COLUMNS = [
    "log_march_impressions",
    "log_march_clicks",
    "march_avg_position",
    "march_ctr",
    "zero_clicks",
    "baseline_eligible",
]

FORBIDDEN_FEATURES = {
    "client_hash_id",
    "content_hash_id",
    "april_impressions",
    "is_declining_label",
    "validation_fold",
    "baseline_score",
}

assert FORBIDDEN_FEATURES.isdisjoint(FEATURE_COLUMNS)

analysis_df[FEATURE_COLUMNS] = analysis_df[
    FEATURE_COLUMNS
].replace([np.inf, -np.inf], np.nan)

estimators = {
    "Logistic Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(max_iter=2000),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=150,
                    max_depth=8,
                    min_samples_leaf=50,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    ),
}

MODEL_SCORE_COLUMNS = {
    "Logistic Regression": "logistic_score",
    "Random Forest": "forest_score",
}

for score_column in MODEL_SCORE_COLUMNS.values():
    analysis_df[score_column] = np.nan

fitted_models = {
    model_name: [] for model_name in estimators
}

def precision_at_k(frame, score_column, k):
    ranked = frame.sort_values(
        [score_column, "content_hash_id"],
        ascending=[False, True],
    )
    return ranked.head(k)["is_declining_label"].mean()

fold_rows = []

for fold_number in range(1, N_SPLITS + 1):
    training_mask = (
        analysis_df["validation_fold"] != fold_number
    )
    validation_mask = (
        analysis_df["validation_fold"] == fold_number
    )

    X_train = analysis_df.loc[
        training_mask, FEATURE_COLUMNS
    ]
    y_train = analysis_df.loc[
        training_mask, "is_declining_label"
    ]
    X_validation = analysis_df.loc[
        validation_mask, FEATURE_COLUMNS
    ]

    for model_name, estimator in estimators.items():
        fitted_estimator = clone(estimator)
        fitted_estimator.fit(X_train, y_train)

        score_column = MODEL_SCORE_COLUMNS[model_name]
        validation_scores = fitted_estimator.predict_proba(
            X_validation
        )[:, 1]

        analysis_df.loc[
            validation_mask, score_column
        ] = validation_scores

        fitted_models[model_name].append(
            {
                "fold": fold_number,
                "estimator": fitted_estimator,
            }
        )

    validation_frame = analysis_df.loc[
        validation_mask
    ].copy()

    assert validation_frame["baseline_eligible"].sum() >= 50

    fold_rows.append(
        {
            "fold": fold_number,
            "base_rate": validation_frame[
                "is_declining_label"
            ].mean(),
            "baseline_precision_at_20": precision_at_k(
                validation_frame, "baseline_score", 20
            ),
            "baseline_precision_at_50": precision_at_k(
                validation_frame, "baseline_score", 50
            ),
            "logistic_precision_at_20": precision_at_k(
                validation_frame, "logistic_score", 20
            ),
            "logistic_precision_at_50": precision_at_k(
                validation_frame, "logistic_score", 50
            ),
            "forest_precision_at_20": precision_at_k(
                validation_frame, "forest_score", 20
            ),
            "forest_precision_at_50": precision_at_k(
                validation_frame, "forest_score", 50
            ),
        }
    )

assert analysis_df["logistic_score"].notna().all()
assert analysis_df["forest_score"].notna().all()

fold_metrics = pd.DataFrame(fold_rows)

fold_display = fold_metrics.copy()
metric_columns = [
    column
    for column in fold_display.columns
    if column != "fold"
]
fold_display[metric_columns] = (
    fold_display[metric_columns] * 100
).round(1)

fold_display = fold_display.rename(
    columns={
        "base_rate": "base_rate_pct",
        "baseline_precision_at_20": "baseline_p20_pct",
        "baseline_precision_at_50": "baseline_p50_pct",
        "logistic_precision_at_20": "logistic_p20_pct",
        "logistic_precision_at_50": "logistic_p50_pct",
        "forest_precision_at_20": "forest_p20_pct",
        "forest_precision_at_50": "forest_p50_pct",
    }
)

comparison_rows = []

for method, prefix in [
    ("Week 4 baseline", "baseline"),
    ("Logistic Regression", "logistic"),
    ("Random Forest", "forest"),
]:
    comparison_rows.append(
        {
            "method": method,
            "mean_base_rate": fold_metrics[
                "base_rate"
            ].mean(),
            "mean_precision_at_20": fold_metrics[
                f"{prefix}_precision_at_20"
            ].mean(),
            "std_precision_at_20": fold_metrics[
                f"{prefix}_precision_at_20"
            ].std(),
            "mean_precision_at_50": fold_metrics[
                f"{prefix}_precision_at_50"
            ].mean(),
            "std_precision_at_50": fold_metrics[
                f"{prefix}_precision_at_50"
            ].std(),
        }
    )

comparison_table = pd.DataFrame(comparison_rows)

comparison_display = comparison_table.copy()

display_columns = [
    "mean_base_rate",
    "mean_precision_at_20",
    "std_precision_at_20",
    "mean_precision_at_50",
    "std_precision_at_50",
]

comparison_display[display_columns] = (
    comparison_display[display_columns] * 100
).round(1)

comparison_display = comparison_display.rename(
    columns={
        "mean_base_rate": "mean_base_rate_pct",
        "mean_precision_at_20": "mean_p20_pct",
        "std_precision_at_20": "p20_std_pp",
        "mean_precision_at_50": "mean_p50_pct",
        "std_precision_at_50": "p50_std_pp",
    }
)

print("Fold results")
display(fold_display)

print("Model-versus-baseline comparison")
display(comparison_display)

Fold results


,fold,base_rate_pct,baseline_p20_pct,baseline_p50_pct,logistic_p20_pct,logistic_p50_pct,forest_p20_pct,forest_p50_pct
0,1,46.3,65.0,56.0,65.0,58.0,65.0,64.0
1,2,67.8,80.0,84.0,80.0,84.0,85.0,90.0
2,3,35.6,70.0,60.0,75.0,68.0,55.0,56.0
3,4,54.4,95.0,78.0,75.0,80.0,50.0,56.0
4,5,51.3,65.0,66.0,65.0,56.0,70.0,68.0


Model-versus-baseline comparison


,method,mean_base_rate_pct,mean_p20_pct,p20_std_pp,mean_p50_pct,p50_std_pp
0,Week 4 baseline,51.1,75.0,12.7,68.8,11.9
1,Logistic Regression,51.1,72.0,6.7,69.2,12.6
2,Random Forest,51.1,65.0,13.7,66.8,14.0


### Comparison result

The Week 4 baseline achieved mean Precision@20 of 75.0%, compared with 72.0% for Logistic Regression. Logistic Regression was only 0.4 percentage points higher at Precision@50: 69.2% versus 68.8%. This difference is too small relative to the variation across folds to count as a meaningful improvement.

Random Forest performed worse and was especially unstable at Precision@20. Its added complexity was not earned.

Logistic Regression still ranked above the 51.1% base rate, but it did not reliably beat the simpler rule. The Week 4 baseline therefore remains the preferred operational method. These are grouped development results, not sealed out-of-sample performance.

---



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Error analysis uses the top 50 because Precision@50 is the project’s primary metric. Top-20 errors are retained as a secondary operational view.

A weak pick is a recommended item that did not cross the 15% April-decline threshold. Feature reliance is assessed using Logistic Regression coefficients and permutation importance at Precision@50. These measures explain model behavior, not causation.

In [10]:
from sklearn.inspection import permutation_importance

def precision_at_50_scorer(estimator, X, y):
    scores = estimator.predict_proba(X)[:, 1]
    top_index = np.argsort(-scores, kind="stable")[:50]
    return float(np.asarray(y)[top_index].mean())

feature_rows = []

for fitted_record in fitted_models["Logistic Regression"]:
    fold_number = fitted_record["fold"]
    estimator = fitted_record["estimator"]
    validation_mask = (
        analysis_df["validation_fold"] == fold_number
    )
    X_validation = analysis_df.loc[
        validation_mask, FEATURE_COLUMNS
    ]
    y_validation = analysis_df.loc[
        validation_mask, "is_declining_label"
    ]

    result = permutation_importance(
        estimator,
        X_validation,
        y_validation,
        scoring=precision_at_50_scorer,
        n_repeats=10,
        random_state=RANDOM_SEED + fold_number,
        n_jobs=1,
    )

    coefficients = estimator.named_steps[
        "model"
    ].coef_[0]

    for feature, coefficient, importance in zip(
        FEATURE_COLUMNS,
        coefficients,
        result.importances_mean,
    ):
        feature_rows.append(
            {
                "fold": fold_number,
                "feature": feature,
                "standardized_coefficient": coefficient,
                "p50_importance": importance,
            }
        )

feature_results = pd.DataFrame(feature_rows)

feature_summary = (
    feature_results.groupby("feature")
    .agg(
        mean_coefficient=(
            "standardized_coefficient", "mean"
        ),
        mean_p50_importance=("p50_importance", "mean"),
        p50_importance_std=("p50_importance", "std"),
    )
    .reset_index()
    .sort_values("mean_p50_importance", ascending=False)
)

feature_display = feature_summary.copy()
feature_display["mean_coefficient"] = (
    feature_display["mean_coefficient"].round(3)
)
feature_display["p50_importance_pp"] = (
    feature_display["mean_p50_importance"] * 100
).round(1)
feature_display["p50_importance_std_pp"] = (
    feature_display["p50_importance_std"] * 100
).round(1)
feature_display = feature_display[
    [
        "feature",
        "mean_coefficient",
        "p50_importance_pp",
        "p50_importance_std_pp",
    ]
]

logistic_ranked = analysis_df.sort_values(
    [
        "validation_fold",
        "logistic_score",
        "content_hash_id",
    ],
    ascending=[True, False, True],
)

analysis_df.loc[
    logistic_ranked.index, "logistic_rank_in_fold"
] = (
    logistic_ranked.groupby("validation_fold")
    .cumcount()
    .to_numpy()
    + 1
)

baseline_ranked = analysis_df.sort_values(
    [
        "validation_fold",
        "baseline_score",
        "content_hash_id",
    ],
    ascending=[True, False, True],
)

analysis_df.loc[
    baseline_ranked.index, "baseline_rank_in_fold"
] = (
    baseline_ranked.groupby("validation_fold")
    .cumcount()
    .to_numpy()
    + 1
)

analysis_df["logistic_top20"] = (
    analysis_df["logistic_rank_in_fold"] <= 20
)
analysis_df["logistic_top50"] = (
    analysis_df["logistic_rank_in_fold"] <= 50
)
analysis_df["baseline_top20"] = (
    analysis_df["baseline_rank_in_fold"] <= 20
)
analysis_df["baseline_top50"] = (
    analysis_df["baseline_rank_in_fold"] <= 50
)
analysis_df["impression_change_pct"] = (
    100
    * (
        analysis_df["april_impressions"]
        - analysis_df["march_impressions"]
    )
    / analysis_df["march_impressions"]
)

top50_frame = analysis_df.loc[
    analysis_df["logistic_top50"]
].copy()

top50_frame["pick_result"] = np.where(
    top50_frame["is_declining_label"] == 1,
    "correct_decline",
    "weak_pick",
)
top50_frame["top20_correct"] = (
    top50_frame["logistic_top20"]
    & (top50_frame["is_declining_label"] == 1)
).astype(int)
top50_frame["top20_weak"] = (
    top50_frame["logistic_top20"]
    & (top50_frame["is_declining_label"] == 0)
).astype(int)
top50_frame["top50_correct"] = (
    top50_frame["is_declining_label"] == 1
).astype(int)
top50_frame["top50_weak"] = (
    top50_frame["is_declining_label"] == 0
).astype(int)

fold_error_summary = (
    top50_frame.groupby("validation_fold")
    .agg(
        top20_correct=("top20_correct", "sum"),
        top20_weak=("top20_weak", "sum"),
        top50_correct=("top50_correct", "sum"),
        top50_weak=("top50_weak", "sum"),
        baseline_top50_overlap=("baseline_top50", "sum"),
    )
    .reset_index()
)

pick_profile = (
    top50_frame.groupby("pick_result")
    .agg(
        picks=("content_hash_id", "size"),
        median_march_impressions=(
            "march_impressions", "median"
        ),
        median_march_clicks=("march_clicks", "median"),
        median_march_ctr=("march_ctr", "median"),
        median_march_position=(
            "march_avg_position", "median"
        ),
        baseline_top50_rate=("baseline_top50", "mean"),
        median_april_change_pct=(
            "impression_change_pct", "median"
        ),
    )
    .reset_index()
)

pick_profile["baseline_top50_rate_pct"] = (
    pick_profile["baseline_top50_rate"] * 100
).round(1)
pick_profile["median_march_ctr"] = (
    pick_profile["median_march_ctr"].round(3)
)
pick_profile["median_march_position"] = (
    pick_profile["median_march_position"].round(2)
)
pick_profile["median_april_change_pct"] = (
    pick_profile["median_april_change_pct"].round(1)
)
pick_profile = pick_profile.drop(
    columns="baseline_top50_rate"
)

weak_picks = top50_frame.loc[
    top50_frame["is_declining_label"] == 0
].copy()

weak_picks["threshold_gap_pp"] = (
    weak_picks["impression_change_pct"] + 15
)

weak_pick_examples = (
    weak_picks.sort_values(
        [
            "validation_fold",
            "logistic_rank_in_fold",
        ]
    )
    .groupby("validation_fold")
    .head(1)
    [
        [
            "validation_fold",
            "logistic_rank_in_fold",
            "logistic_top20",
            "logistic_score",
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "baseline_eligible",
            "baseline_top50",
            "impression_change_pct",
            "threshold_gap_pp",
        ]
    ]
    .copy()
)

weak_pick_examples["logistic_score"] = (
    weak_pick_examples["logistic_score"].round(3)
)
weak_pick_examples["march_ctr"] = (
    weak_pick_examples["march_ctr"].round(3)
)
weak_pick_examples["march_avg_position"] = (
    weak_pick_examples["march_avg_position"].round(2)
)
weak_pick_examples["impression_change_pct"] = (
    weak_pick_examples["impression_change_pct"].round(1)
)
weak_pick_examples["threshold_gap_pp"] = (
    weak_pick_examples["threshold_gap_pp"].round(1)
)

print("Logistic Regression feature reliance at P@50")
display(feature_display)

print("Top-20 and top-50 errors by fold")
display(fold_error_summary)

print("Top-50 correct-pick versus weak-pick profile")
display(pick_profile)

weak_pick_examples = weak_pick_examples.reset_index(drop=True)
weak_pick_examples["logistic_rank_in_fold"] = (
    weak_pick_examples["logistic_rank_in_fold"].astype(int)
)

print("Highest-ranked weak pick in each fold")
display(weak_pick_examples)

Logistic Regression feature reliance at P@50


,feature,mean_coefficient,p50_importance_pp,p50_importance_std_pp
1,log_march_clicks,-0.640,24.2,11.4
2,log_march_impressions,0.668,18.1,14.2
3,march_avg_position,-0.086,3.0,2.8
4,march_ctr,0.051,2.6,2.8
5,zero_clicks,-0.036,2.2,2.3
0,baseline_eligible,0.021,1.7,2.7


Top-20 and top-50 errors by fold


,validation_fold,top20_correct,top20_weak,top50_correct,top50_weak,baseline_top50_overlap
0,1,13,7,29,21,43
1,2,16,4,42,8,42
2,3,15,5,34,16,41
3,4,15,5,40,10,18
4,5,13,7,28,22,29


Top-50 correct-pick versus weak-pick profile


,pick_result,picks,median_march_impressions,median_march_clicks,median_march_ctr,median_march_position,median_april_change_pct,baseline_top50_rate_pct
0,correct_decline,173,6396.0,0.0,0.0,6.64,-55.1,70.5
1,weak_pick,77,5707.0,0.0,0.0,8.52,10.0,66.2


Highest-ranked weak pick in each fold


,validation_fold,logistic_rank_in_fold,logistic_top20,logistic_score,march_impressions,march_clicks,march_ctr,march_avg_position,baseline_eligible,baseline_top50,impression_change_pct,threshold_gap_pp
0,1,1,True,0.896,134984.0,1.0,0.001,2.69,0,False,-5.1,9.9
1,2,7,True,0.776,9312.0,0.0,0.000,5.83,1,True,55.1,70.1
2,3,7,True,0.828,8220.0,0.0,0.000,18.57,1,True,10.5,25.5
3,4,6,True,0.849,30834.0,0.0,0.000,40.57,0,False,-2.4,12.6
4,5,6,True,0.851,10209.0,0.0,0.000,29.09,0,False,-2.6,12.4


### Error findings

Log clicks and log impressions were the strongest features. Their opposite coefficient signs indicate that high exposure combined with few clicks raised the model score. Average position ranked third, while CTR, zero clicks, and baseline eligibility added little independent information. Importance varied across folds, so these relationships were not equally stable across client groups.

Logistic Regression made 173 correct and 77 weak picks across the five top-50 queues. Correct and weak picks looked similar in March: both groups had median zero clicks and zero CTR. This suggests that the available March snapshot cannot reliably separate continued decline from temporary low-click patterns.

Three high-ranked errors illustrate the limitation. A fold-1 rank-1 item declined by only 5.1%, missing the label threshold by 9.9 percentage points. A fold-2 rank-7 item increased by 55.1%, despite meeting the baseline rule. A fold-4 rank-6 item declined by only 2.4% and ranked outside the baseline rule because its average position was worse than 20.

The model scores describe ranking order, not calibrated probabilities. These errors do not show that any March feature caused the later outcome.

---



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.